# Prediction and Process Monitoring

Projects new observations onto existing PCA/PLS models and evaluates them against training-set control limits.

In [1]:
import pandas as pd
import numpy as np
import pyphi.calc as phi
import pyphi.plots as pp
from bokeh.io import output_notebook
output_notebook(hide_banner=True)
import pyphi.plots as _ppmod
_ppmod.output_file = lambda *a, **kw: None


Will be using the NEOS server in the absence of IPOPT and GAMS


## Build Training Models

In [2]:
features    = pd.read_excel('../data/Automobiles PLS.xlsx', 'Features',
                            na_values=np.nan, engine='openpyxl')
performance = pd.read_excel('../data/Automobiles PLS.xlsx', 'Performance',
                            na_values=np.nan, engine='openpyxl')

# Hold out last 10 rows as 'new' observations
X_train = features.iloc[:-10].reset_index(drop=True)
X_new   = features.iloc[-10:].reset_index(drop=True)
Y_train = performance.iloc[:-10].reset_index(drop=True)
Y_new   = performance.iloc[-10:].reset_index(drop=True)

pcaobj = phi.pca(X_train, 3)
plsobj = phi.pls(X_train, Y_train, 3)
print('PCA model built on', X_train.shape[0], 'observations')
print('PLS model built on', X_train.shape[0], 'observations')


phi.pca using NIPALS executed on: 2026-03-26 23:50:59.238991
# Iterations for PC #1:  10
# Iterations for PC #2:  8
# Iterations for PC #3:  35
--------------------------------------------------------------
PC #      Eig        R2X       sum(R2X) 
PC #1:      3.872    0.777     0.777
PC #2:      0.820    0.165     0.942
PC #3:      0.171    0.031     0.973
--------------------------------------------------------------
phi.pls using NIPALS executed on: 2026-03-26 23:50:59.257949
# Iterations for LV #1:  5
# Iterations for LV #2:  28
# Iterations for LV #3:  9
--------------------------------------------------------------
LV #     Eig       R2X       sum(R2X)   R2Y       sum(R2Y)
LV #1:    3.852    0.776     0.776      0.548     0.548
LV #2:    0.135    0.032     0.808      0.142     0.690
LV #3:    0.801    0.161     0.969      0.015     0.705
--------------------------------------------------------------
PCA model built on 396 observations
PLS model built on 396 observations


## PCA Prediction on New Observations

`pca_pred` returns a dict with `Tnew`, `speX`, `T2` and reconstructed `Xhat`.

In [3]:
pca_pred = phi.pca_pred(X_new, pcaobj)
print('Keys:', list(pca_pred.keys()))
print('Tnew shape:', pca_pred['Tnew'].shape)
print('speX:', pca_pred['speX'].ravel())


Keys: ['Xhat', 'Tnew', 'speX', 'T2']
Tnew shape: (10, 3)
speX: [0.16059491 0.26729104 0.08405727 0.2827682  0.05230345 0.19246574
 0.10305493 0.07932312 0.04288061 0.07501479]


## PLS Prediction on New Observations

`pls_pred` returns `Yhat` (predicted Y) plus the same SPE/T² diagnostics.

In [4]:
pls_pred = phi.pls_pred(X_new, plsobj)
print('Keys:', list(pls_pred.keys()))
print('Yhat shape:', pls_pred['Yhat'].shape)
Y_actual = np.array(Y_new.values[:, 1:]).astype(float)
resid = Y_actual - pls_pred['Yhat']
print('Prediction residuals (first 5 rows):\n', resid[:5])


Keys: ['Yhat', 'Xhat', 'Tnew', 'speX', 'T2']
Yhat shape: (10, 2)
Prediction residuals (first 5 rows):
 [[ 1.59060498 -3.0427357 ]
 [ 0.05359124 11.87539745]
 [-1.18919026 -5.32249427]
 [ 0.5098547  -5.7921236 ]
 [-2.95461983  3.70156952]]


## Hotelling's T² for New Observations

`hott2` returns the T² statistic for each observation; flag observations outside the 95/99% limits.

In [5]:
t2_new = phi.hott2(pcaobj, Xnew=X_new)
print('T2new:', t2_new)
print('T2_lim95:', pcaobj['T2_lim95'], '  T2_lim99:', pcaobj['T2_lim99'])
above_99 = t2_new > pcaobj['T2_lim99']
print('Observations above 99% limit:', X_new.iloc[:, 0].values[above_99])


T2new: [3.42254713 5.50880537 3.78044596 3.50550246 3.29001475 3.66790641
 3.41080446 3.29269997 3.11804575 3.26999667]
T2_lim95: 7.942964728992131   T2_lim99: 11.583093354502477
Observations above 99% limit: <StringArray>
[]
Length: 0, dtype: str


## Interpreting SPE and T² Violations

- **T² high, SPE normal**: observation is within the model hyperplane but at an extreme score location.
- **SPE high**: observation has structure not captured by the model (possible process fault or new condition).
- **Both high**: severe outlier.

In [6]:
spe_new = pca_pred['speX'].ravel()
t2_new  = phi.hott2(pcaobj, Xnew=X_new)
for i, obs in enumerate(X_new.iloc[:, 0].values):
    spe_flag = '!' if spe_new[i] > pcaobj['speX_lim99'] else ' '
    t2_flag  = '!' if t2_new[i]  > pcaobj['T2_lim99']  else ' '
    print(f'{obs:20s}  SPE={spe_new[i]:.3f}{spe_flag}  T2={t2_new[i]:.3f}{t2_flag}')


Car397                SPE=0.161   T2=3.423 
Car398                SPE=0.267   T2=5.509 
Car399                SPE=0.084   T2=3.780 
Car400                SPE=0.283   T2=3.506 
Car401                SPE=0.052   T2=3.290 
Car402                SPE=0.192   T2=3.668 
Car403                SPE=0.103   T2=3.411 
Car404                SPE=0.079   T2=3.293 
Car405                SPE=0.043   T2=3.118 
Car406                SPE=0.075   T2=3.270 
